In [6]:
import numpy as np
import sympy as sp
import re
import matplotlib.pyplot as plt

def check_allowed_char(num):
    allowed_char = r'^[0-9*.+-/]+$'
    if re.match(allowed_char, num):
        #print("Input has only allowed characters.")
        return 1
    else:
        #print("You have used characters that we do not recognize as numbers.")
        return 0

def extract_pieces(num):
    # Define a regular expression pattern.
    pattern = r'\d+|[^0-9]'
    # Find all pieces matching the pattern
    pieces = re.findall(pattern, num)
    return pieces

def check_pieces_1(pieces):
    S = []
    for i in range(len(pieces) - 2):
        if ((pieces[i] == '+' and pieces[i+1] == '*') or 
            (pieces[i] == '-' and pieces[i+1] == '*') or
            (pieces[i] == '*' and pieces[i+1] == '/') or
            (pieces[i] == '/' and pieces[i+1] == '*') or
            (pieces[i] == '.' and pieces[i+1] == '.') or
            (pieces[i] == '+' and pieces[i+1] == '/') or
            (pieces[i] == '-' and pieces[i+1] == '/') or
            (pieces[i] == '/' and pieces[i+1] == '/')):
            S.append(1)  
    if np.sum(S) >= 1:
        #print("You have entered a string that cannot be interpreted numerically.")
        return 0
    else:
        #print("You have passed check_pieces_1")
        return 1

def check_pieces_2_endpts(pieces):
    if ((pieces[len(pieces)-1] == '+') or (pieces[len(pieces)-1] == '*') or 
        (pieces[len(pieces)-1] == '-') or (pieces[len(pieces)-1] == '/') or
        (pieces[0] == '/') or (pieces[0] == '*')):
        #print("You have begun or ended your string with notation that cannot be interpreted numerically.")
        return 0
    else:
        #print("You have passed check_pieces_2_endpts")
        return 1

def check_pieces_3_star(pieces):
    T = []
    for i in range(len(pieces) - 3):
        if ((pieces[i] == '*') and (pieces[i+1] == '*') and 
                         (pieces[i+2] == '*')):
            T.append(1)
    if np.sum(T) >= 1:
        #print("You have entered ***, which cannot be interpreted numerically.")
        return 0
    else:
        #print("You have passed check_pieces_3_star")
        return 1
            

def check_pieces_4_decimal(pieces):
    U = []
    for i in range(len(pieces) - 3): 
        if (pieces[i].isdigit() and pieces[i+1] == '.' and pieces[i+2].isdigit() 
            and pieces[i+3] == '.'):
            U.append(1)
    if np.sum(U) >= 1: 
        #print("ERROR:You have entered two decimal points in a single numeric string.")
        return 0
    else: 
        #print("You have passed check_pieces_4_decimal")
        return 1

def nat_gen(entry):
    pieces = extract_pieces(entry)
    if ((check_allowed_char(entry) == 1) and (check_pieces_1(pieces) == 1) 
        and (check_pieces_2_endpts(pieces) == 1) and (check_pieces_3_star(pieces) == 1) and 
        (check_pieces_4_decimal(pieces) == 1)):
        expression = ''.join(pieces)
        real_exp = sp.sympify(expression)
        if ((real_exp % 1 != 0) or (real_exp < 0)):
            raise ValueError("Your number is not a positive integer.")
        else:
            #print("Your entry is interpretable as a valid positive integer.")
            return real_exp
    else: 
        raise ValueError("Your string failed to be interpreted numerically.")

def float_gen(entry):
    pieces = extract_pieces(entry)
    if ((check_allowed_char(entry) == 1) and (check_pieces_1(pieces) == 1) 
        and (check_pieces_2_endpts(pieces) == 1) and (check_pieces_3_star(pieces) == 1) and 
        (check_pieces_4_decimal(pieces) == 1)):
        expression = ''.join(pieces)
        real_exp = sp.sympify(expression)
        #print("Your entry is interpretable as a valid real number.")
        return real_exp
    else:
        raise ValueError("Your string failed to be interpreted numerically.")

def stepsize(x, p, q):
    if x == 0:
        raise ValueError("Invalid Operation: You are attempting to divide by 0.")
    elif ((x != 0) and (p == q)):
        return 0
    else:
        return abs(p - q) / (100 * x)

#The user inputs the number of segments
numseg = input("Please enter the number of segments you want partition [0,1] into.")
n = nat_gen(numseg)
print(f"Your chosen number of segments is {n}")

#Define Intervals
x = input("Please enter the value a (or) b for your interval [a, b] on which f is defined.")
y = input("Please enter the value a (or) b for your interval [a, b] on which f is defined.")
a = float_gen(x)
b = float_gen(y)
print(f"Your first chosen interval endpoint is {a}")
print(f"Your second chosen interval endpoint is {b}")

#We set the stepsize s as a fraction of a hundredth of the length of the prescribed interval.
z = input("Please enter the number the step size must be multiplied by to get 1/100 * |x - y|.")
z = nat_gen(z)

s = stepsize(z, a, b) 
print(f"Your chosen stepsize is {s}")

Please enter the number of segments you want partition [0,1] into. 5


Your chosen number of segments is 5


Please enter the value a (or) b for your interval [a, b] on which f is defined. 1
Please enter the value a (or) b for your interval [a, b] on which f is defined. 3


Your first chosen interval endpoint is 1
Your second chosen interval endpoint is 3


Please enter the number the step size must be multiplied by to get 1/100 * |x - y|. 50


Your chosen stepsize is 1/2500


In [8]:
#We create an initial partition, which we choose to be the partition splitting the interval [min{a, b}, max{a, b}]
#chosen by the user into n equal parts, where n is the number of segments (also chosen by the user).

init_partition = [0] * (n + 1)
init_partition[0] = min(a, b)
init_partition[n] = max(a, b)

for i in range(1, n):
    init_partition[i] = init_partition[0] + ((init_partition[n] - init_partition[0]) * i / n)

print(init_partition)

[1, 7/5, 9/5, 11/5, 13/5, 3]


In [14]:
#Allow the user to build the univariate function
import numpy as np
import sympy as sp
import re
import matplotlib.pyplot as plt

def fn_allowed_char(fn):
    fn_allowed_char = re.compile(r'^[a-zA-Z0-9*+-/^.()]+$')
    if re.match(fn_allowed_char, fn):
        #print("Input function has only allowed characters.")
        return 1
    else:
        #print("You have used characters that we do not recognize as defining a valid function.")
        return 0

def extract_fn_pieces(fn):
    fn = fn.replace('^', '**')
    # Define a regular expression pattern.
    pattern = re.compile(r'sin|cos|tan|exp|log|[a-zA-Z]|[+\-*/^.()]|\d+')
    # Find all pieces matching the pattern
    pieces = re.findall(pattern, fn)
    return pieces

def add_stars(pieces):
    pieces_starred = []
    symb = ['sin', 'cos', 'tan', 'log', 'exp']
    for i in range(len(pieces)-1):
        pieces_starred.append(pieces[i])
        if (((pieces[i].isalpha()) and (pieces[i] not in symb) and (pieces[i+1].isdigit())) or 
            ((pieces[i].isdigit()) and (pieces[i+1].isalpha())) or 
            (((pieces[i].isalpha())) and (pieces[i+1].isalpha()) and (pieces[i] not in symb)) or 
            ((pieces[i].isalpha()) and (pieces[i] not in symb) and (pieces[i+1] == '.')) or 
            ((pieces[i] == '.') and (pieces[i+1].isalpha())) or 
            ((pieces[i] == ')') and ((pieces[i+1] == 'sin') or (pieces[i+1] == 'exp') or (pieces[i+1] == 'cos') or (pieces[i+1] == 'tan') or (pieces[i+1] == 'log') or (pieces[i+1] == '('))) or 
            ((pieces[i].isdigit()) and (pieces[i+1] == '(')) or
            ((pieces[i].isalpha()) and (pieces[i] not in symb) and (pieces[i+1] == '(')) or 
            ((pieces[i] == ')') and (pieces[i+1].isalpha())) or
            ((pieces[i] == ')') and (pieces[i+1].isdigit()))):
            pieces_starred.append('*')
    pieces_starred.append(pieces[len(pieces)-1])
    return pieces_starred

def count_alph(pieces):
    alphabets = []
    alph_collect = ['sin', 'cos', 'tan', 'log', 'exp']
    for i in range(len(pieces)):
        if pieces[i].isalpha() and pieces[i] not in alph_collect:
            alphabets.append(1)
            alph_collect.append(pieces[i])
        else:
            alphabets.append(0)
    if np.sum(alphabets) >= 3:
        #print("We are only equipped to handle functions of 2 variables or less.")
        return 0
    else:
        #print("You entered a function of 2 variables or less.")
        return 1

def br_nobr_rules(pieces):
    error_count = 0
    for i in range(len(pieces)-1):
        if ((pieces[i] == '(') and ((pieces[i+1] == '/') or (pieces[i+1] == '*') or (pieces[i+1] == '^') or (pieces[i+1] == ')'))):
            print("You are breaking a bracket rule and therefore, your string cannot be interpreted sensibly.")
            error_count+=1
        elif (((pieces[i] == 'sin') or (pieces[i] == 'cos') or (pieces[i] == 'tan') 
               or (pieces[i] == 'log') or (pieces[i] == 'exp')) and (pieces[i+1] != '(')):
            print("You must use '(' immediately following 'sin', 'cos', 'tan', 'exp' or 'log'.")
            error_count+=1
        elif ((pieces[len(pieces) - 1] == 'sin') or (pieces[len(pieces) - 1] == 'cos') or
        (pieces[len(pieces) - 1] == 'tan') or (pieces[len(pieces) - 1] == 'log') or 
        (pieces[len(pieces) - 1] == 'exp') or (pieces[len(pieces) - 1] == '(')):
            print("You may not end your string with sin, cos, tan, log, or exp.")
            error_count+=1
        else:
            error_count+=0
    if error_count >= 1:
        print(f"You have made {error_count} error(s) in your string between bracket and non-bracket pieces.")
        return 0
    else:
        print("Your string does not contain errors between brackets and non-bracket pieces.")
        return 1

def br_br_balance(pieces):
    def count_br_op(i):
        count_brack_open = 0
        if (1 <= i <= len(pieces)):
            for j in range(i):
                if (pieces[j] == '('):
                    count_brack_open+=1
                else:
                    count_brack_open+=0
        return count_brack_open
    def count_br_cl(i):
        count_brack_closed = 0
        if (1 <= i <= len(pieces)):
            for j in range(i):
                if (pieces[j] == ')'):
                    count_brack_closed+=1
                else:
                    count_brack_closed+=0
        return count_brack_closed
    br_br_fault = 0
    for i in range(1, len(pieces)+1):
        if (i <= len(pieces) - 1) and (count_br_op(i) < count_br_cl(i)):
            print(".")
            br_br_fault+=1
        elif (i == len(pieces)) and (count_br_op(i) > count_br_cl(i)):
            br_br_fault+=1
        else:
            br_br_fault+=0
    if br_br_fault >= 1:
        print("You have closed a bracket without opening it first.")
        return 0
    else:
        print("Your open-closed bracket pairs occur in an acceptable sequence.")
        return 1

def user_fn_input(entry):
    pieces_raw = extract_fn_pieces(entry)
    pieces = add_stars(pieces_raw)
    if ((count_alph(pieces) == 1) and (fn_allowed_char(entry) == 1) and (check_pieces_1(pieces) == 1) 
        and (check_pieces_2_endpts(pieces) == 1) and (check_pieces_3_star(pieces) == 1) and 
        (check_pieces_4_decimal(pieces) == 1) and (br_nobr_rules(pieces) == 1) and (br_br_balance(pieces) == 1)):
        fn_str_expr = ''.join(pieces)
        fn_expr = sp.sympify(fn_str_expr)
        fn_var = sorted(fn_expr.free_symbols, key = lambda symbol: symbol.name)
        fnct = sp.lambdify(fn_var, fn_expr, 'numpy')
        #print("Your entry is interpretable as a valid function.")      
        if len(fn_var) == 0:
            def f():
                return fnct()
        elif len(fn_var) == 1:
            def f(x):
                return fnct(x)
        elif len(fn_var) == 2:
            def f(x, y):
                return fnct(x, y)
        return f, fn_var
    else:
        raise ValueError("Your string failed to be interpreted as a function of 2 variables or less.")

#Code for user input
user_fn = input("Enter a function (NO SPACES): ")
user_pieces_raw = extract_fn_pieces(user_fn)
user_pieces = add_stars(user_pieces_raw)
#print(fn_allowed_char(user_fn), count_alph(user_pieces), check_pieces_1(user_pieces), check_pieces_2_endpts(user_pieces), check_pieces_3_star(user_pieces), 
 #     check_pieces_4_decimal(user_pieces), br_nobr_rules(user_pieces), br_br_balance(user_pieces))
user_f, var = user_fn_input(user_fn)

# Test the function with sample input
if len(var) == 0:
    result = user_f()
if len(var) == 1:
    result = user_f(1)
elif len(var) == 2:
    result = user_f(1, 2) 
print(f"Function result: {result}")
#print(user_pieces)

Enter a function:  x^2+3


Your string does not contain errors between brackets and non-bracket pieces.
Your open-closed bracket pairs occur in an acceptable sequence.
Function result: 4
